# Quanto custa morar em Lucas do Rio Verde (MT): o mercado imobiliário por dentro

**Autora:** Thamires Martins da Silva
**Fonte dos dados:** Anúncios de venda de 13 imobiliárias e portais locais + Índice FipeZap (contexto regional)
**Ferramentas:** Python (pandas, matplotlib)

---

## Contexto

Terceiro e último projeto de uma análise em três camadas sobre habitação em Lucas do Rio Verde:

- **Camada 1** — projetou a *demanda* futura por moradia (crescimento populacional).
- **Camada 2** — mediu a *oferta* real de habitação de interesse social.
- **Camada 3 (este notebook)** — mede o *preço*: quanto custa comprar um imóvel em Lucas hoje, e onde.

A lógica fecha o ciclo. Se a população cresce rápido (Camada 1) e a oferta não acompanha (Camada 2), a teoria econômica prevê **pressão sobre os preços**. Esta camada testa isso com dado de mercado.

> **Pergunta de negócio:** Quanto custa morar em Lucas do Rio Verde hoje, como o preço varia entre bairros, e como se compara ao de uma capital?

## Sobre o dataset

Não existe índice público de preço imobiliário para Lucas — o FipeZap cobre só grandes cidades (em MT, apenas Cuiabá). Por isso o dataset foi **construído a partir de anúncios de venda** coletados dos sites das imobiliárias locais, com preço, área construída e bairro de cada imóvel. São **236 anúncios de 25 bairros, coletados de 13 fontes independentes** (Domus, MGF, Nossa Casa, Agro+, Benedetti, Volpato, Tati Griebler, Viva, Dinâmica, Modelo e Ideal, além dos portais ZAP e Loft).

**Critério de qualidade na coleta:** o preço por m² só faz sentido sobre a **área construída**. Fontes cujos anúncios informavam apenas a área do terreno foram descartadas (caso da Líder Imobiliária); da Dinâmica, cujos cards misturavam área construída e de terreno nas casas, entraram apenas os apartamentos (onde a área é privativa, por definição construída). Modelo e Ideal, que separam área construída de terreno, entraram com casas e apartamentos.

**Limitações assumidas:** (1) retrato do momento (outubro/2026), não série histórica — anúncios antigos não ficam arquivados; (2) preço de anúncio é preço *pedido*, não de fechamento; (3) como várias imobiliárias de Lucas trabalham com carteira aberta, um mesmo imóvel pode, em princípio, aparecer em duas fontes (verificação por área+preço não encontrou duplicatas óbvias). Imóveis atípicos (mansões de luxo, micro-investimento) são marcados e separados da análise principal.

## 1. Configuração e carga

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

AZUL_ESC='#1a3a5c'; AZUL='#2c7fb8'; TEAL='#7fcdbb'; LARANJA='#e8703a'; CINZA='#8a9299'

df = pd.read_csv(Path.cwd()/'precos_imoveis_lucas.csv', sep=';')

# Verifica duplicatas exatas e remove-as como proteção adicional.
duplicatas_exatas = int(df.duplicated().sum())
if duplicatas_exatas:
    df = df.drop_duplicates().copy()

# Combinações iguais podem corresponder ao mesmo imóvel em fontes diferentes.
# São sinalizadas para revisão manual, sem remoção automática.
chave_candidata = ['bairro', 'tipo', 'area_m2', 'preco_total', 'preco_m2']
candidatos_duplicacao = df[df.duplicated(subset=chave_candidata, keep=False)].sort_values(chave_candidata)

print(f'Duplicatas exatas removidas ao carregar: {duplicatas_exatas}')
print(f'Total de anúncios após verificação: {len(df)}')
print(f'Registros com possível sobreposição entre fontes: {len(candidatos_duplicacao)}')
if not candidatos_duplicacao.empty:
    print(candidatos_duplicacao[chave_candidata + ['fonte']].to_string(index=False))
print(f'Bairros nomeados: {df[df["bairro"]!="Não informado"]["bairro"].nunique()}')
print(f'Fontes: {df["fonte"].value_counts().to_dict()}')
print()
print(df[['bairro','tipo','area_m2','preco_total','preco_m2','fonte']].head(6).to_string(index=False))

## 2. Tratamento: separando o mercado típico

Imóveis **atípicos** — mansões de alto padrão e micro-apartamentos de investimento — distorcem a média e não representam o que uma família comum encontra. A análise principal usa só os típicos.

**Qualidade da base:** três linhas inteiras repetidas (mesmo bairro, tipo, área, preço, fonte e data) foram removidas antes desta versão pública. O arquivo publicado tem 233 registros. Combinações iguais de bairro/tipo/área/preço em fontes diferentes ficam como candidatas para inspeção manual; não são removidas automaticamente, pois podem ser anúncios espelhados ou imóveis distintos com características iguais.

In [ ]:
df['atip'] = df['atipico']=='sim'
tip = df[~df['atip']].copy()
print(f'Total:    {len(df)}')
print(f'  Tipicos:  {len(tip)}  (analise principal)')
print(f'  Atipicos: {df["atip"].sum()}  (luxo e investimento, separados)')
print()
print('--- Qualidade (tipicos) ---')
print(f'Nulos preco_m2: {tip["preco_m2"].isna().sum()}')
print(f'Area: {tip["area_m2"].min():.0f} a {tip["area_m2"].max():.0f} m2')
print(f'Casas: {(tip["tipo"]=="casa").sum()} | Apartamentos: {(tip["tipo"]=="apartamento").sum()}')

## 3. Quanto custa o metro quadrado em Lucas

In [ ]:
pm2 = tip['preco_m2']
print('=== PRECO POR M2 (tipicos) ===')
print(f'  Media:   R$ {pm2.mean():>6,.0f}/m2'.replace(',','.'))
print(f'  Mediana: R$ {pm2.median():>6,.0f}/m2'.replace(',','.'))
print(f'  Min-Max: R$ {pm2.min():,.0f} a R$ {pm2.max():,.0f}'.replace(',','.'))

In [ ]:
fig,ax=plt.subplots(figsize=(11,5.5))
ax.hist(pm2,bins=14,color=AZUL,edgecolor='white',zorder=3)
ax.axvline(pm2.median(),color=LARANJA,linestyle='--',linewidth=2.5,zorder=4,
           label=f'Mediana: R$ {pm2.median():,.0f}/m2'.replace(',','.'))
ax.set_title(f'Distribuicao do preco por m2 em Lucas do Rio Verde\nImoveis tipicos (n={len(pm2)})',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Preco por m2 (R$)'); ax.set_ylabel('Numero de imoveis'); ax.legend(fontsize=11)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
plt.tight_layout(); plt.show()

### Leitura

O metro quadrado em Lucas do Rio Verde gira em torno de **R\$ 6.222 (mediana)**, com a metade central dos imóveis entre cerca de R\$ 4.600 e R\$ 7.500/m². É um patamar alto para uma cidade de porte médio no interior — a seção 6 mostra o quanto, comparando com a capital.

## 4. O mapa de preços: quanto custa cada bairro

Com 25 bairros na amostra, dá para traçar a geografia de preços da cidade. Mostramos os bairros com pelo menos 5 anúncios, para que a média de cada um tenha robustez (anúncios sem bairro informado ficam de fora deste recorte).

In [ ]:
por_bairro = tip[tip['bairro']!='Não informado'].groupby('bairro')['preco_m2'].agg(['count','mean']).reset_index()
por_bairro = por_bairro[por_bairro['count']>=5].sort_values('mean',ascending=False)
print('=== PRECO/M2 POR BAIRRO (5+ anuncios) ===')
for _,r in por_bairro.iterrows():
    print(f'  {r["bairro"]:22} n={int(r["count"]):2}  R$ {r["mean"]:>6,.0f}/m2'.replace(',','.'))

In [ ]:
fig,ax=plt.subplots(figsize=(11,6.5))
pb = por_bairro.sort_values('mean')
cores=[LARANJA if v==pb['mean'].max() else (TEAL if v==pb['mean'].min() else AZUL) for v in pb['mean']]
ax.barh(pb['bairro'],pb['mean'],color=cores,zorder=3)
for i,(v,n) in enumerate(zip(pb['mean'],pb['count'])):
    ax.annotate(f'R$ {v:,.0f}'.replace(',','.'),(v,i),textcoords='offset points',
                xytext=(5,0),va='center',fontsize=10,fontweight='bold',color=AZUL_ESC)
ax.set_title('O mapa de precos de Lucas do Rio Verde\nPreco medio por m2, por bairro',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_xlabel('Preco por m2 (R$)')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='x',alpha=.25,zorder=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
plt.tight_layout(); plt.show()

### Leitura: uma cidade de muitos preços

A geografia de preços é nítida. No topo, **Parque das Emas (~R\$ 8.000/m²), Bandeirantes (~R\$ 7.700/m²) e Parque dos Buritis (~R\$ 7.200/m²)** — os bairros valorizados, de apartamentos novos e casas de padrão mais alto. Na base, **Jardim Imperial, Jardim Primavera, Cidade Nova e Rio Verde (~R\$ 3.600 a 4.250/m²)** — bairros mais populares e afastados.

A diferença é enorme: o m² no bairro mais caro custa **cerca de 120% mais** que no mais acessível — mais que o dobro. Para quem compra, isso significa que o mesmo orçamento rende mais que o dobro de área dependendo da região — a escolha do bairro é a decisão de maior impacto no bolso.

## 5. Casa ou apartamento, e o que cabe no orçamento

In [ ]:
pt = tip.groupby('tipo')['preco_m2'].agg(['count','mean','median'])
print('=== PRECO/M2 POR TIPO ===')
for t,r in pt.iterrows():
    print(f'  {t.capitalize():12} n={int(r["count"]):2}  media R$ {r["mean"]:,.0f}  mediana R$ {r["median"]:,.0f}'.replace(',','.'))
print()
pr = tip['preco_total']
print('=== FAIXA DE PRECO TOTAL ===')
print(f'  Mais acessivel: R$ {pr.min():>10,.0f}'.replace(',','.'))
print(f'  Mediana:        R$ {pr.median():>10,.0f}'.replace(',','.'))
print(f'  Mais caro:      R$ {pr.max():>10,.0f}'.replace(',','.'))

In [ ]:
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5.5))
# Esq: casa vs apto
tipos=['casa','apartamento']; medias=[pt.loc[t,'mean'] for t in tipos]
ax1.bar(['Casa','Apartamento'],medias,color=[TEAL,AZUL],width=.5,zorder=3)
for x,y in enumerate(medias):
    ax1.annotate(f'R$ {y:,.0f}'.replace(',','.'),(x,y),textcoords='offset points',
                 xytext=(0,8),ha='center',fontsize=12,fontweight='bold',color=AZUL_ESC)
ax1.set_title('Apartamento custa mais por m2',fontsize=12,color=AZUL_ESC,pad=12)
ax1.set_ylabel('Preco medio por m2 (R$)'); ax1.set_ylim(0,max(medias)*1.2)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
ax1.grid(True,axis='y',alpha=.25,zorder=0)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
# Dir: scatter preco x area
for t,cor,lbl in [('casa',TEAL,'Casa'),('apartamento',AZUL,'Apartamento')]:
    s=tip[tip['tipo']==t]
    ax2.scatter(s['area_m2'],s['preco_total'],color=cor,s=70,alpha=.75,
                edgecolor='white',linewidth=1,label=lbl,zorder=3)
ax2.set_title('Preco x area',fontsize=12,color=AZUL_ESC,pad=12)
ax2.set_xlabel('Area (m2)'); ax2.set_ylabel('Preco total'); ax2.legend(fontsize=10)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.grid(True,alpha=.25,zorder=0)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{x/1000:,.0f}k'.replace(',','.')))
plt.tight_layout(); plt.show()

### Leitura

**Apartamento custa cerca de 43% mais por m²** que casa (R\$ 7.556 vs R\$ 5.273, medianas). Faz sentido: apartamentos concentram-se nos bairros valorizados e trazem lazer de condomínio embutido; a casa dilui o custo ao espalhar área construída.

O imóvel típico sai por volta de **R\$ 540 mil**, com a entrada no mercado perto de **R\$ 205 mil** (apartamentos compactos e casas populares, muitas ligadas a programas habitacionais como os da Camada 2). Quem busca espaço pelo mesmo dinheiro tende à casa; quem prioriza localização e comodidades, ao apartamento.

## 6. Lucas vs. a capital: o soco de contexto

Lucas tem ~95 mil habitantes. Seria de esperar que fosse bem mais barata que Cuiabá, a capital com mais de 650 mil. Usamos o FipeZap (abril/2026) como referência oficial.

In [ ]:
lucas=tip['preco_m2'].median(); cuiaba=6931; brasil=9769
print('=== PRECO/M2: LUCAS vs CAPITAL vs BRASIL ===')
print(f'  Lucas do Rio Verde (mediana): R$ {lucas:>6,.0f}/m2'.replace(',','.'))
print(f'  Cuiaba - capital (FipeZap):   R$ {cuiaba:>6,.0f}/m2'.replace(',','.'))
print(f'  Brasil - 56 cidades (FipeZap):R$ {brasil:>6,.0f}/m2'.replace(',','.'))
print()
print(f'  Lucas vs Cuiaba:  {(lucas/cuiaba-1)*100:+.0f}%')
print(f'  Lucas vs Brasil:  {(lucas/brasil-1)*100:+.0f}%')

In [ ]:
fig,ax=plt.subplots(figsize=(11,5.5))
loc=['Lucas do\nRio Verde','Cuiaba\n(capital)','Brasil\n(56 cidades)']
val=[lucas,cuiaba,brasil]
ax.bar(loc,val,color=[LARANJA,AZUL,CINZA],width=.55,zorder=3)
for x,y in enumerate(val):
    ax.annotate(f'R$ {y:,.0f}'.replace(',','.'),(x,y),textcoords='offset points',
                xytext=(0,8),ha='center',fontsize=13,fontweight='bold',color=AZUL_ESC)
ax.set_title('Lucas do Rio Verde cobra 90% do m2 da capital do estado\nPreco por m2 - residencial',
             fontsize=13,color=AZUL_ESC,pad=15)
ax.set_ylabel('Preco por m2 (R$)'); ax.set_ylim(0,max(val)*1.18)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(True,axis='y',alpha=.25,zorder=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,p:f'{int(x):,}'.replace(',','.')))
plt.tight_layout(); plt.show()

### Leitura: o preço confirma a pressão

Este é o resultado central. Lucas do Rio Verde, uma cidade do interior com cerca de 95 mil habitantes, cobra um metro quadrado a **apenas ~10% abaixo da capital do estado** — ou seja, cerca de 90% do preço de Cuiabá, que tem quase sete vezes a sua população. Para uma cidade desse porte, estar tão perto do nível de preço da capital é um sinal forte de mercado pressionado.

É o que as duas camadas anteriores previam. A **Camada 1** mostrou população quase quintuplicada e ainda crescendo. A **Camada 2** mostrou oferta de moradia aquém do ritmo. Muita gente disputando pouca oferta empurra o preço para cima — e é o que o mercado confirma. As três camadas contam uma história coerente: **demanda alta + oferta insuficiente = preços próximos aos de capital numa cidade do interior.**

## 7. Limitações

1. **Retrato do momento, não série histórica.** Dados de outubro/2026. Anúncios antigos não ficam arquivados, então não é possível reconstruir a evolução temporal de preços a partir de anúncios atuais.
2. **Preço de anúncio, não de fechamento.** É o valor pedido, geralmente acima do preço real de venda.
3. **Possível sobreposição entre fontes.** Como imobiliárias locais trabalham com carteira aberta, um mesmo imóvel pode aparecer em mais de uma fonte. A verificação por combinação de área e preço não encontrou duplicatas óbvias, mas o risco não é zero.
4. **Área construída depende do que cada fonte informa.** Fontes que só traziam área de terreno foram descartadas; da Dinâmica entraram só apartamentos. Ainda assim, parte das áreas vem do próprio anúncio e não de medição independente.
5. **Médias por bairro com amostras variadas.** Bairros com 5+ anúncios entram no mapa de preços; alguns têm 5–6 registros e são indicativos, não medições definitivas.
6. **Comparação regional com fontes distintas.** Lucas vem de anúncios; Cuiabá e Brasil, do FipeZap (metodologia própria). A comparação indica ordem de grandeza, não medição idêntica.

## 8. Síntese executiva

Comprar um imóvel em Lucas do Rio Verde custa, hoje, cerca de **R\$ 6.222 por m² (mediana)**, com o imóvel típico saindo por volta de **R\$ 540 mil** e a entrada no mercado perto de **R\$ 205 mil**. Apartamentos custam cerca de 43% mais por m² que casas.

A cidade tem uma **geografia de preços marcada**: do Parque das Emas e Bandeirantes (~R\$ 7.700 a 8.000/m²) ao Jardim Imperial e Jardim Primavera (~R\$ 3.600 a 4.200/m²), o m² varia mais de 120% conforme o bairro — a escolha da região é a decisão de maior peso no orçamento de quem compra.

O achado que dá sentido ao projeto é a comparação regional: **uma cidade do interior cobra um metro quadrado a cerca de 90% do valor da capital do estado.** Não é coincidência — é a consequência direta do que as duas camadas anteriores mostraram. A população cresce rápido (Camada 1), a oferta não acompanha (Camada 2), e o preço responde (Camada 3).

Para quem acompanha o mercado local — construtoras, investidores, famílias — a leitura é clara: Lucas do Rio Verde já não é uma cidade barata. O preço reflete uma demanda estrutural que a oferta ainda não atendeu, e enquanto esse descompasso persistir, a pressão sobre os preços tende a continuar.

---

*Dataset com 236 anúncios (188 típicos) de 13 fontes independentes, coletados em outubro/2026 dos sites das imobiliárias de Lucas do Rio Verde. Base, código e as três camadas da análise no GitHub.*